# LongRoPE diagnostic — does the |V|=128 prompt cross Phi-3.5's 4096-token seam?

**Purpose:** confirm or kill the hypothesis in `PORTABLE-HANDOFF-v2.md` v11 / `DRAFT_v2.md` §4.3
for the `|V|=128` anomaly in E9. This is a **tokenizer-only** check — no model weights loaded,
no generation, no GPU. Runs in under a minute on CPU.

**Kaggle setup:** Settings -> Accelerator -> **None (CPU)**. Settings -> Internet -> **ON**
(needed to fetch the BFCL data + the tokenizer, both small). Run All.

This reconstructs the *exact* candidate sets E9 actually evaluated at `|V|=128` — same
`MASTER_SEED`, same `task_id`s, same seed derivation, same candidate-rendering code as the
canonical notebook — then tokenizes the resulting prompts with Phi-3.5-mini-instruct's own
tokenizer and reports the token count against its documented `original_max_position_embeddings
= 4096`.


In [ ]:
!pip install -q transformers==5.16.1

In [ ]:
import os, re, json, random, hashlib, urllib.request, ssl

def derive_seed(*parts) -> int:
    key = "||".join(str(p) for p in parts).encode("utf-8")
    return int(hashlib.sha256(key).hexdigest()[:16], 16) % (2 ** 31 - 1)

MASTER_SEED = 20260822   # identical to every run in this project -- do not change
BFCL_DESC_CHARS = 110    # identical to E9's config.json

BFCL_BASE = ("https://raw.githubusercontent.com/ShishirPatil/gorilla/main/"
             "berkeley-function-call-leaderboard/bfcl_eval/data")

def fetch(url, dest):
    ctx = ssl.create_default_context()
    req = urllib.request.Request(url, headers={"User-Agent": "svc-research"})
    with urllib.request.urlopen(req, timeout=240, context=ctx) as r:
        data = r.read()
    with open(dest, "wb") as fh:
        fh.write(data)
    return dest

def load_jsonl(path):
    out = []
    with open(path, encoding="utf-8") as fh:
        for line in fh:
            line = line.strip()
            if line:
                out.append(json.loads(line))
    return out

os.makedirs("/kaggle/working/bfcl_data", exist_ok=True)
NEEDED = ["BFCL_v4_multiple.json", "BFCL_v4_live_multiple.json",
          "BFCL_v4_simple_python.json", "BFCL_v4_live_simple.json",
          "BFCL_v4_parallel_multiple.json"]
paths = {}
for name in NEEDED:
    dest = f"/kaggle/working/bfcl_data/{name}"
    if not os.path.exists(dest):
        fetch(f"{BFCL_BASE}/{name}", dest)
    paths[name] = dest
for name in ["BFCL_v4_multiple.json", "BFCL_v4_live_multiple.json"]:
    dest = f"/kaggle/working/bfcl_data/ANS_{name}"
    if not os.path.exists(dest):
        fetch(f"{BFCL_BASE}/possible_answer/{name}", dest)
    paths["ANS_" + name] = dest
print("data ready:", list(paths.keys()))

In [ ]:
# ---- verbatim from the canonical BFCL_CORE cell: case/pool construction ----
def _question_text(q):
    if isinstance(q, str):
        return q
    parts, stack = [], [q]
    while stack:
        cur = stack.pop(0)
        if isinstance(cur, dict):
            if cur.get("role") == "user" and cur.get("content"):
                parts.append(str(cur["content"]))
        elif isinstance(cur, (list, tuple)):
            stack = list(cur) + stack
    return " ".join(parts).strip()

def _gt_name(ans_row):
    gt = ans_row.get("ground_truth")
    if isinstance(gt, list) and len(gt) == 1 and isinstance(gt[0], dict) and len(gt[0]) == 1:
        return next(iter(gt[0].keys()))
    return None

def build_cases(paths, datasets, min_v=2):
    cases = []
    for name in datasets:
        if name not in paths or ("ANS_" + name) not in paths:
            continue
        rows = load_jsonl(paths[name])
        answers = {r["id"]: r for r in load_jsonl(paths["ANS_" + name])}
        for r in rows:
            rid = r.get("id")
            fns = r.get("function") or []
            ans = answers.get(rid)
            if ans is None or len(fns) < min_v:
                continue
            gt = _gt_name(ans)
            if not gt:
                continue
            names = [f.get("name") for f in fns]
            if gt not in names:
                continue
            cases.append(dict(case_id=str(rid), dataset=name,
                               question=_question_text(r.get("question")),
                               functions=fns, gt_name=gt, native_v=len(fns)))
    return cases

def function_pool_from_files(paths, datasets):
    seen, pool = set(), []
    for name in datasets:
        if name not in paths:
            continue
        for r in load_jsonl(paths[name]):
            for f in (r.get("function") or []):
                nm = f.get("name")
                if nm and nm not in seen:
                    seen.add(nm)
                    pool.append(f)
    return pool

def make_candidate_set(case, v_level, rng, pool):
    native = list(case["functions"])
    gt = case["gt_name"]
    keep = [f for f in native if f.get("name") == gt]
    others = [f for f in native if f.get("name") != gt]
    rng.shuffle(others)
    if v_level <= len(native):
        out = keep + others[:max(0, v_level - 1)]
    else:
        have = {f.get("name") for f in native}
        extra = [f for f in pool if f.get("name") not in have]
        rng.shuffle(extra)
        out = keep + others + extra[:max(0, v_level - len(native))]
    rng.shuffle(out)
    return out

def render_candidates(functions, desc_chars):
    lines = []
    for i, f in enumerate(functions):
        d = re.sub(r"\s+", " ", str(f.get("description", ""))).strip()
        if len(d) > desc_chars:
            d = d[:desc_chars].rstrip() + "..."
        lines.append(f"{i}. {f.get('name')} - {d}")
    return "\n".join(lines)

BFCL_COMMON = """User request: {question}"""
BFCL_SUFFIX_CONSTRAINED = """
Candidate functions:
{candidates}

Reply with ONLY the number of the function you choose."""

def render_bfcl_prompt(question, functions, desc_chars):
    common = BFCL_COMMON.format(question=question)
    return common + BFCL_SUFFIX_CONSTRAINED.format(
        candidates=render_candidates(functions, desc_chars))

cases = build_cases(paths, ["BFCL_v4_multiple.json", "BFCL_v4_live_multiple.json"], min_v=2)
pool = function_pool_from_files(paths, NEEDED)
by_id = {c["case_id"]: c for c in cases}
print(f"{len(cases)} cases, pool={len(pool)}")

In [ ]:
# ---- the ACTUAL task_ids E9 evaluated at |V|=128 (from your real per_case.csv) ----
REAL_V128_CASES = [
    ("live_multiple_229-103-0", 0),
    ("live_multiple_229-103-0", 1),
    ("live_multiple_652-161-20", 0),
]

SYSTEM_PROMPT_CHARS = len(
    "You select exactly one function to call. Answer with the selection only. Do not explain."
)  # this is the system message; apply_chat_template adds a bit more overhead on top

for case_id, seed_idx in REAL_V128_CASES:
    case = by_id[case_id]
    cand_seed = derive_seed(MASTER_SEED, seed_idx, case_id, "CAND")
    cand_rng = random.Random(cand_seed)
    functions = make_candidate_set(case, 128, cand_rng, pool)
    prompt = render_bfcl_prompt(case["question"], functions, BFCL_DESC_CHARS)
    globals().setdefault("PROMPTS", []).append((case_id, seed_idx, prompt))
    print(f"{case_id} seed={seed_idx}: {len(prompt):,} chars (candidate block only, "
          f"excludes system prompt + chat template overhead)")

In [ ]:
# ---- the actual tokenizer check -- this is the number that matters ----
from transformers import AutoTokenizer

tok = AutoTokenizer.from_pretrained("microsoft/Phi-3.5-mini-instruct")

SYSTEM_PROMPT = ("You select exactly one function to call. "
                  "Answer with the selection only. Do not explain.")

print(f"{'case_id':30s} {'seed':>4s}  {'raw_chars':>9s}  {'tokens (full chat-templated prompt)':>36s}")
for case_id, seed_idx, prompt in PROMPTS:
    msgs = [{"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": prompt}]
    text = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    n_tok = len(tok(text)["input_ids"])
    print(f"{case_id:30s} {seed_idx:4d}  {len(text):9,}  {n_tok:36,}")

print()
print("Phi-3.5-mini-instruct config.original_max_position_embeddings = 4096 (LongRoPE seam)")
print("If the token counts above are within roughly +/-300 of 4096, that supports the")
print("hypothesis in DRAFT_v2.md section 4.3. If they are well clear of 4096 in either")
print("direction, the LongRoPE explanation is NOT supported and should be removed from the")
print("draft -- report the |V|=128 collapse as an unexplained anomaly instead.")